# 3. Quatre modèles supervisés candidats et choix du meilleur

| Modèle | Type | Pourquoi lui |
| --- | --- | --- |
| Random Forest | supervisé | apprend sur des attaques étiquetées ; robuste, explicable avec SHAP |
| XGBoost | supervisé | boosting, souvent le plus précis sur des données tabulaires ; explicable avec SHAP |
| Régression Logistique | supervisé | baseline **linéaire** simple et interprétable ; sert de point de comparaison (les arbres devraient la dépasser) |
| MLP | supervisé | réseau de neurones ; apporte une famille **non-arbre** au comparatif |

**Protocole.** Les quatre modèles apprennent sur les lignes des blocs d'entraînement
(normales **et** attaques). Le seuil d'alerte de chaque modèle est choisi sur la
**validation** (F1 maximal), puis appliqué tel quel au **test**.

> Les modèles **non supervisés** (Isolation Forest, One-Class SVM) et la référence
> statistique ont été **retirés** : le comparatif ne porte que sur des modèles supervisés.


In [ ]:
%%capture
import sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend").is_dir())
sys.path[:0] = [str(ROOT / "backend"), str(ROOT / "backend" / "scripts")]
import build_dataset as bd
import transmission_real as tr
import transmission_models as tm
import transmission_study as ts
from app.api.v1 import phase2
from app.core import paths

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (11, 3.5)
SEED = tr.SEED
COLS = {"weather": "temperature", "fridge": "fridge_temperature", "thermostat": "current_temperature"}
EPISODES_PATH = paths.PROCESSED_DIR / "episodes_transmission_reel.json"
MODEL_DIR = paths.MODELS_DIR / "transmission_reel"

In [ ]:
df = pd.read_csv(tr.DATASET_PATH, parse_dates=["datetime"], keep_default_na=False)
df["garde"] = df["garde"].astype(str).str.lower().eq("true")
df, feats = tm.engineer(df)
episodes = json.load(open(EPISODES_PATH))

y = df["label"].to_numpy().astype(int)
# Le découpage train/val/test n'est pas dans le dataset : on le calcule ICI, à
# l'entraînement, de façon NON aléatoire (70/15/15 par blocs d'une heure).
split = tr.block_split(len(df), run=0)
garde = df["garde"].to_numpy()
train_all = np.where((split == "train") & ~garde)[0]
train_norm = train_all[y[train_all] == 0]
val_idx = np.where((split == "val") & ~garde)[0]
test_idx = np.where((split == "test") & ~garde)[0]
X = df[feats].to_numpy(float)
from sklearn.preprocessing import StandardScaler
X_s = StandardScaler().fit(X[train_norm]).transform(X)
print(f"train {len(train_all):,} lignes ({len(train_norm):,} normales) | val {len(val_idx):,} | test {len(test_idx):,}")


## 3.0 Validation croisée stratifiée (protocole demandé)

- `cross_val_score` fait passer au modèle **5 examens** (5 folds) : 4 morceaux pour apprendre,
  1 pour noter, en tournant → 5 notes, dont on retient la **moyenne** et l'**écart-type**.
- `StratifiedKFold` (le **stratify**) garantit que chaque fold garde le même % d'attaques (~33 %).
- `scoring="f1"` est **obligatoire** : l'accuracy mentirait sur un dataset déséquilibré.
  Un modèle qui crierait « normal » à tout le temps aurait ~**67 %** d'accuracy (le train est
  à ~33 % d'attaques). C'est pour ça qu'on note au **F1** et non à l'accuracy.



In [ ]:
# ===== VALIDATION CROISÉE : 5 EXAMENS AU LIEU D'UN =====
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)  # "stratify"

X_tr, y_tr = X[train_all], y[train_all]
pos_tr = int(y_tr.sum())
# MLP n'a pas de class_weight : on équilibre les classes à la main (même logique)
poids_tr = np.where(y_tr == 1, len(y_tr) / (2 * pos_tr), len(y_tr) / (2 * (len(y_tr) - pos_tr)))

modeles_cv = {
    "Régression Logistique": (LogisticRegression(max_iter=1000, class_weight="balanced",
                                                 random_state=SEED), X_s),
    "Random Forest": (RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                             class_weight="balanced_subsample",
                                             random_state=SEED, n_jobs=-1), X),
    "XGBoost": (XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8,
                              colsample_bytree=0.8, scale_pos_weight=(len(y_tr) - pos_tr) / pos_tr,
                              eval_metric="logloss", random_state=SEED, n_jobs=8, verbosity=0), X),
    "MLP": (MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, early_stopping=True,
                          random_state=SEED), X_s),
}

for nom, (mod, Xm) in modeles_cv.items():
    notes = cross_val_score(mod, Xm[train_all], y_tr, cv=cv, scoring="f1")
    print(f"{nom:<22} F1 moyen {notes.mean():.3f} +/- {notes.std():.3f}   notes : {notes.round(2)}")

## 3.0bis Validation croisée honnête (par groupes) — la vraie comparaison

**Le problème.** Nos attaques durent 12 à 20 minutes. À 5 secondes par mesure, cela fait
beaucoup de lignes presque identiques. Avec un découpage **au hasard** (`StratifiedKFold`),
une partie de la **même attaque** tombe dans le train *et* dans le test : le modèle la
**reconnaît** et obtient un F1 de ~0,99. C'est un score **trop beau** (fuite temporelle).

**L'image.** C'est comme réviser avec *exactement* les questions de l'examen : on a 20/20
sans rien prouver.

**La solution.** On garde chaque attaque **entière** d'un seul côté : soit entièrement dans
le train, soit entièrement dans le test. C'est le principe de `GroupKFold` (le groupe = une
attaque complète). `StratifiedGroupKFold` fait **les deux** : il garde le même % d'attaques
(stratify, demandé par le prof) **et** ne coupe jamais une attaque.

**Ce qu'on obtient.** Le score **baisse** (vers 0,7) mais devient **honnête** — et il
**sépare** enfin les modèles.


In [ ]:
# ===== VALIDATION CROISÉE HONNÊTE (par groupes) =====
# Règle d'or : une attaque reste ENTIÈRE d'un seul côté (jamais coupée entre train et test).
from sklearn.model_selection import StratifiedGroupKFold

# 1) Une étiquette par ligne :
#    - toutes les lignes d'une même attaque -> le même groupe ("attaque_k")
#    - les lignes normales -> un groupe par heure ("normale_<heure>")
groups = np.empty(len(df), dtype=object)
heure = df["datetime"].dt.floor("h").astype(str).to_numpy()
i, k = 0, 0
while i < len(y):
    if y[i] == 1:                       # bloc contigu d'attaque = UNE attaque entière
        j = i
        while j + 1 < len(y) and y[j + 1] == 1:
            j += 1
        k += 1
        groups[i:j + 1] = f"attaque_{k}"
        i = j + 1
    else:
        groups[i] = "normale_" + heure[i]
        i += 1

n_att = len([g for g in set(groups) if str(g).startswith("attaque_")])
print(f"Groupes créés : {n_att} attaques entières + {len(set(groups)) - n_att} blocs d'heures normales")

# 2) Validation croisée par groupes : Stratified (garde le % d'attaques) + Group (ne coupe pas les attaques)
cv_groupes = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
groups_train = groups[train_all]

print()
print(f"{'Modèle':<24}{'F1 naïf (au hasard)':>24}{'F1 honnête (par groupes)':>28}")
print("-" * 76)
for nom, (mod, Xm) in modeles_cv.items():
    naif = cross_val_score(mod, Xm[train_all], y_tr, cv=cv, scoring="f1")
    honnete = cross_val_score(mod, Xm[train_all], y_tr, cv=cv_groupes,
                              groups=groups_train, scoring="f1")
    print(f"{nom:<24}{naif.mean():>15.3f} +/-{naif.std():.3f}{honnete.mean():>17.3f} +/-{honnete.std():.3f}")

print()
print("À retenir : la version honnête donne un score plus bas mais VRAI.")
print("Elle désigne le même gagnant que la validation et le test (Random Forest).")


## 3.1 Entraînement des modèles supervisés

Ce que cette cellule NE fait PAS

❌ Elle ne choisit pas le seuil. 

❌ Elle ne compare pas les modèles. 

❌ Elle ne donne pas de résultat final.

Elle fait une seule chose : entraîner + calculer les scores.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

scores = {}

# --- Random Forest (supervisé : a appris sur des attaques étiquetées) ---
# n_estimators=300                : 300 arbres votent (vote majoritaire)
# min_samples_leaf=5              : une conclusion doit reposer sur >= 5 exemples (évite le surapprentissage)
# class_weight=balanced_subsample : les attaques sont moins nombreuses -> on leur donne plus de poids
# random_state=SEED               : on fixe le hasard
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced_subsample",
                            random_state=SEED, n_jobs=-1).fit(X[train_all], y[train_all])
scores["Random Forest"] = rf.predict_proba(X)[:, 1]           # % d'arbres qui votent attaque (0 à 1)

# --- XGBoost (supervisé : les arbres corrigent les erreurs des précédents) ---
# n_estimators=400       : 400 arbres en série (chaque arbre corrige le précédent)
# max_depth=6            : au plus 6 questions enchaînées par arbre
# learning_rate=0.05     : chaque arbre ne corrige que 5% de l'erreur -> apprentissage prudent
# subsample=0.8          : chaque arbre voit 80% des lignes au hasard -> diversité
# colsample_bytree=0.8   : chaque arbre voit 80% des colonnes (capteurs) au hasard -> diversité
# scale_pos_weight=...   : les attaques sont moins nombreuses -> on leur donne plus de poids
pos = int(y[train_all].sum())
xgb = XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                    scale_pos_weight=(len(train_all) - pos) / pos, eval_metric="logloss",
                    random_state=SEED, n_jobs=8, verbosity=0).fit(X[train_all], y[train_all])
scores["XGBoost"] = xgb.predict_proba(X)[:, 1]                 # probabilité d'attaque (0 à 1)
print("2 modèles supervisés entraînés (les 2 autres arrivent juste après)")

# ===== AFFICHER LE SCORE DE CHAQUE MODÈLE =====
from IPython.display import display

# Les 8 premiers scores de chaque modèle, côte à côte
apercu = pd.DataFrame({nom: np.round(s[:8], 4) for nom, s in scores.items()})
apercu.index.name = "ligne"
display(apercu)

# Le résumé de l'échelle de chaque modèle
resume = pd.DataFrame({nom: {"minimum": round(s.min(), 4),
                             "maximum": round(s.max(), 4),
                             "moyenne": round(s.mean(), 4)}
                       for nom, s in scores.items()})
display(resume)


### 2 nouveaux modèles supervisés (protocole identique aux autres)

- **Régression Logistique** : baseline **linéaire interprétable** (les arbres devraient la dépasser
  car la frontière normal/attaque n'est pas droite → c'est le rôle d'un baseline).
- **MLP** (réseau de neurones) : apporte une famille non-arbre au comparatif.
- On obtient au final **4 modèles supervisés** comparés sur le même protocole.


In [ ]:
# --- Régression Logistique (linéaire, interprétable) ---
# X_s (standardisé) est obligatoire pour les modèles linéaires et les neurones
lr = LogisticRegression(max_iter=1000, class_weight="balanced",
                        random_state=SEED).fit(X_s[train_all], y[train_all])
scores["Régression Logistique"] = lr.predict_proba(X_s)[:, 1]

# --- MLP (réseau de neurones) ---
# early_stopping=True : arrête quand la progression s'arrête (anti-surapprentissage)
# sample_weight=poids_tr : équilibre attaques/normales (MLP n'a pas de class_weight)
mlp = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, early_stopping=True,
                    random_state=SEED).fit(X_s[train_all], y[train_all], sample_weight=poids_tr)
scores["MLP"] = mlp.predict_proba(X_s)[:, 1]

print("Modeles dans scores :", list(scores))

## 3.2 Évaluation sur le test (répétition 0)

Prend les scores et choisit un seuil pour chaque modèle (sur la validation).
Fait passer l'examen (test) et affiche un tableau : AUC, précision, rappel, F1, fausses alertes/h, épisodes détectés, délai

In [ ]:
# Outils de mesure pour noter les modèles (AUC, etc.)
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

# rows = liste qui va recevoir une ligne de résultats par modèle
# preds = dictionnaire qui va stocker les décisions (True = alerte) de chaque modèle
rows, preds = [], {}

for nom, s in scores.items():          # on passe en revue les 4 modèles, un par un
    # 1) On cherche le seuil d'alerte sur la VALIDATION (jamais sur le test !)
    thr = phase2._best_f1_threshold(y[val_idx], s[val_idx])
    # 2) On transforme les scores en décisions : True (alerte) si score >= seuil
    preds[nom] = s >= thr
    # 3) On compte les bons coups et les erreurs de ce modèle sur le TEST
    r = tr.pool([tr.count_run(preds[nom], df, test_idx, episodes, "test")])
    # 4) On ajoute une ligne de résultats pour ce modèle
    rows.append({"modèle": nom,
                 "AUC": round(roc_auc_score(y[test_idx], s[test_idx]), 3),   # qualité du tri (0.5 = hasard, 1 = parfait)
                 "précision": r["precision"],       # quand j'alerte, ai-je raison ?
                 "rappel": r["recall"],             # sur toutes les attaques, combien j'en attrape ?
                 "F1": r["f1"],                     # équilibre précision/rappel (la note principale)
                 "fausses alertes / h": r["fausses_alarmes_evenements_par_heure"],   # plus bas = mieux
                 "épisodes détectés": f"{r['episodes_detectes']}/{r['episodes']}",   # ex. 6/6
                 "délai médian (min)": r["delai_median_min"]})                      # temps de réaction
# On affiche le tout dans un tableau : une ligne par modèle
pd.DataFrame(rows).set_index("modèle")

### Matrice de confusion — à quoi ça sert ?


montre où le modèle se trompe (fausses alertes vs attaques ratées). 

La matrice de confusion est un **tableau 2×2** qui compare la **prédiction** du modèle et la **vraie réponse** :

|  | le modèle dit *attaque* | le modèle dit *normal* |
| --- | --- | --- |
| c'est vraiment une attaque | **Vrai positif** : attaque bien détectée | **Faux négatif** : attaque ratée (le pire !) |
| c'est vraiment normal | **Faux positif** : fausse alerte | **Vrai négatif** : normal reconnu |

- **Précision** = Vrai positif / (Vrai positif + Faux positif) → « quand j'alerte, est-ce que j'ai raison ? »
- **Rappel** = Vrai positif / (Vrai positif + Faux négatif) → « sur toutes les attaques, combien en ai-je trouvées ? »

On la regarde pour comprendre **où** le modèle se trompe : trop de fausses alertes, ou au contraire des attaques
qu'il laisse passer. Un seul chiffre comme le F1 ne donne que la note finale, pas le type d'erreur.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

if "preds" not in dir():   # cellule exécutée seule : il faut d'abord avoir lancé le notebook plus haut
    raise NameError("Exécute d'abord les cellules 3.1 et 3.2 (les données, les modèles et le tableau de résultats), puis relance cette cellule.")

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, (nom, pred) in zip(axes.ravel(), preds.items()):
    cm = confusion_matrix(y[test_idx], pred[test_idx])
    ConfusionMatrixDisplay(cm, display_labels=["normal", "attaque"]).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(nom, fontsize=10)
plt.suptitle("Matrices de confusion (test, répétition 0)", y=1.02, fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(6, 5))
for nom, s in scores.items():
    p, r, _ = precision_recall_curve(y[test_idx], s[test_idx])
    plt.plot(r, p, label=f"{nom} (AP {average_precision_score(y[test_idx], s[test_idx]):.2f})")
plt.xlabel("rappel"); plt.ylabel("précision"); plt.title("Courbes précision-rappel (test, répétition 0)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()

In [ ]:
types = df["attack_type"].to_numpy()
par_scenario = {}
for nom, pred in preds.items():
    m = np.zeros(len(df), bool); m[test_idx] = True
    par_scenario[nom] = {sc: round(100 * (pred & m & (types == sc)).sum() / max(1, (m & (types == sc)).sum()), 1)
                         for sc in tr.ATTACK_NAMES if (m & (types == sc)).any()}
pd.DataFrame(par_scenario).rename_axis("% des lignes d'attaque signalées (test, répétition 0)")

### 3.2bis Tableau de synthèse « produits pharmaceutiques »

Ordre de priorité pour des produits thermosensibles :
**rappel** (ne rien rater — chaque attaque ratée = lots potentiellement perdus) >
**F1** (note officielle) > **fausses alertes/h** (fatigue d'alarme) >
**délai médian** (alerter avant dégradation) > **épisodes détectés** > **ROC-AUC**.

Seuil choisi par modèle sur la **validation** (F1 maximal), appliqué tel quel au **test**.
La colonne **FN** = attaques ratées → la case la plus critique de la matrice de confusion.

In [ ]:
from sklearn.metrics import (precision_recall_curve, confusion_matrix,
                             recall_score, f1_score, roc_auc_score)

heures_test = len(test_idx) / 720          # 720 mesures = 1 heure
eps_test = [e for e in episodes if e["split"] == "test"]
resultats = []

for nom, s in scores.items():
    # seuil = F1 maximal sur la VALIDATION (jamais choisi sur le train ni sur le test)
    prec_r, rec_r, sc_r = precision_recall_curve(y[val_idx], s[val_idx])
    f1_r = 2 * prec_r * rec_r / (prec_r + rec_r + 1e-9)
    seuil = float(sc_r[np.argmax(f1_r)])

    pred = (s[test_idx] >= seuil).astype(int)
    tn, fp, fn, tp = confusion_matrix(y[test_idx], pred).ravel()

    delais, detectes = [], 0
    for ep in eps_test:
        seg = s[ep["start"]:ep["end"]] >= seuil
        if seg.any():
            detectes += 1
            delais.append(int(np.argmax(seg)) * 5 / 60)   # minutes avant la 1re alerte

    resultats.append({
        "rappel": round(recall_score(y[test_idx], pred), 3),
        "F1": round(f1_score(y[test_idx], pred), 3),
        "fausses alertes/h": round(fp / heures_test, 1),
        "delai median (min)": round(float(np.median(delais)), 1) if delais else None,
        "episodes detectes": f"{detectes}/{len(eps_test)}",
        "ROC-AUC": round(roc_auc_score(y[test_idx], s[test_idx]), 3),
        "FN (attaques ratees)": int(fn),
        "seuil": round(seuil, 3),
    })

pd.DataFrame(resultats, index=list(scores)).sort_values("F1", ascending=False)

Une seule répétition ne contient que 9 épisodes de test : trop peu pour juger. Le script
`backend/scripts/transmission_models.py` rejoue l'étude 9 fois (rôles des heures et scénarios tournants)
et cumule les résultats : **~80 épisodes de test**, répartis sur les 7 scénarios.


## 3.3 Résultat cumulé sur les 9 répétitions

In [ ]:
fichier = MODEL_DIR / "comparaison_modeles.json"
if fichier.exists():
    res = json.load(open(fichier, encoding="utf-8"))["resultats"]
    cumul = pd.DataFrame({r["nom"]: {"précision": r["precision"], "rappel": r["recall"], "F1": r["f1"],
                                     "F1 validation": r["val_f1"], "fausses alertes / h": r["fausses_alarmes_evenements_par_heure"],
                                     "épisodes détectés": f"{r['episodes_detectes']}/{r['episodes']}"}
                          for r in res.values()}).T.sort_values("F1 validation", ascending=False)
    display(cumul)
    par_sc = pd.DataFrame({r["nom"]: {sc: v["rate"] for sc, v in r["per_attack"].items()} for r in res.values()})
    display(par_sc.rename_axis("% des lignes d'attaque signalées (test cumulé)"))
else:
    print("Lancer d'abord :  cd backend && .venv/bin/python scripts/transmission_models.py")

**Choix.** Le **Random Forest (supervisé)** était le meilleur de l'étude de référence (sur la
validation comme sur le test). Relance le script `transmission_models.py` pour obtenir les chiffres
à jour après ce changement (4 modèles supervisés, ~33 % d'attaques).

**Pourquoi lui.** Il repère des **formes** qui trahissent l'attaquant : la valeur ne bouge plus (gel),
la fenêtre se répète (rejeu), les valeurs sont trop régulières (injection nominale), ou le **niveau
moyen lissé** a baissé (biais, dérive). Le **XGBoost** arrive juste derrière.

**Limite restante.** La **dérive lente** (`Injection_derive`, rampe de 0 à −2 °C) reste la plus
difficile : le décalage se construit doucement et ressemble à un refroidissement normal.

Limite à garder en tête : un modèle supervisé reconnaît les attaques qui ressemblent à celles de son
entraînement. Une attaque d'un type nouveau ne serait pas forcément vue.


In [ ]:
import joblib
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(rf, MODEL_DIR / "random_forest.pkl")
json.dump(feats, open(MODEL_DIR / "random_forest_features.json", "w"), indent=1)
print("Random Forest enregistré :", MODEL_DIR / "random_forest.pkl")